In [8]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
import lightgbm as lgb 

In [9]:
# import sys
# print(sys.executable)

In [10]:
# import sys
# !"{sys.executable}" -m pip install lightgbm


In [11]:
train = pd.read_csv(r"../Data/train_red.csv")
holidays = pd.read_csv("../Data/holidays_events.csv")
items = pd.read_csv("../Data/items.csv")
oil = pd.read_csv("../Data/oil.csv")
stores = pd.read_csv("../Data/stores.csv")
transactions = pd.read_csv(r"../Data/transactions.csv")

C:\Users\Yug Patel\AppData\Local\Temp\ipykernel_4836\2225389715.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(r"../Data/train_red.csv")


In [12]:
train.shape

(1067263, 6)

In [13]:
holidays.shape

(350, 6)

In [14]:
items.shape

(4100, 4)

In [15]:
oil.shape

(1218, 2)

In [16]:
stores.shape

(54, 5)

In [17]:
transactions.shape

(83488, 3)

In [18]:
train.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,22,2013-01-01,25,119187,1.0,NaN
1,54,2013-01-01,25,205387,1.0,NaN
2,89,2013-01-01,25,265257,5.0,NaN
3,131,2013-01-01,25,319094,1.0,NaN
4,142,2013-01-01,25,358096,1.0,NaN


In [19]:
train['onpromotion'].value_counts()

onpromotion
False    822286
True      67097
Name: count, dtype: int64

In [20]:
train['onpromotion'] = train['onpromotion'].fillna(False)

C:\Users\Yug Patel\AppData\Local\Temp\ipykernel_4836\3626677686.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['onpromotion'] = train['onpromotion'].fillna(False)


In [21]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [22]:
items.head()

,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


In [23]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [24]:
oil.set_index('date')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()  
oil = oil.reset_index()


In [25]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [26]:
transactions.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [27]:
holidays['type'].value_counts()

type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

In [28]:
holidays.isnull().sum()

date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

In [29]:
items.isnull().sum()

item_nbr      0
family        0
class         0
perishable    0
dtype: int64

In [30]:
stores.isnull().sum()

store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

In [31]:
transactions.isnull().sum()

date            0
store_nbr       0
transactions    0
dtype: int64

In [32]:
train["date"] = pd.to_datetime(train["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
transactions['date'] = pd.to_datetime(transactions["date"])
oil['date'] = pd.to_datetime(oil['date'])

In [33]:
train.info()
holidays.info()
transactions.info()
items.info()
oil.info()
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067263 entries, 0 to 1067262
Data columns (total 6 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   id           1067263 non-null  int64         
 1   date         1067263 non-null  datetime64[ns]
 2   store_nbr    1067263 non-null  int64         
 3   item_nbr     1067263 non-null  int64         
 4   unit_sales   1067263 non-null  float64       
 5   onpromotion  1067263 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(1), int64(3)
memory usage: 41.7 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[ns]
 1   type         350 non-null    object        
 2   locale       350 non-null    object        
 3   locale_name  350 non-null    object        
 4 

In [34]:
df = train.merge(items, on="item_nbr", how='left')

In [35]:
df = df.merge(stores, on="store_nbr", how="left")

In [36]:
df = df.merge(oil[['date', 'dcoilwtico']], on='date', how='left')

In [37]:
df = df.merge(transactions, on=["store_nbr", "date"], how="left")

In [38]:
df = df.merge(holidays, on='date', how='left')

In [39]:
df = df.rename(columns={"type_x": "store_type", "type_y": "holiday_type"})

In [41]:
df.to_csv("../Data/data.csv",index=False)